In [0]:
storage_account_name = "silveradlsstorage"
gold_base = f"abfss://gold@{storage_account_name}.dfs.core.windows.net"
gold_path = f"{gold_base}/retail_dw"
sql_server = "pavansqlserver123.database.windows.net"
sql_database = "sql-db"
secret_scope = "retail-adls-kv-scope"
storage_account_fqdn = f"{storage_account_name}.dfs.core.windows.net"


def get_secret_or_fail(scope_name, secret_key):
    try:
        return dbutils.secrets.get(scope_name, secret_key)
    except Exception as err:
        raise RuntimeError(
            f"Missing secret '{secret_key}' in scope '{scope_name}'. "
            "Add it to the linked Azure Key Vault and rerun Cell 1."
        ) from err


sql_user = get_secret_or_fail(secret_scope, "azure-sql-user")
sql_password = get_secret_or_fail(secret_scope, "azure-sql-password")
tenant_id = get_secret_or_fail(secret_scope, "adls-sp-tenant-id")

try:
    spark.conf.unset(f"fs.azure.account.key.{storage_account_fqdn}")
except Exception:
    pass

spark.conf.set(f"fs.azure.account.auth.type.{storage_account_fqdn}", "OAuth")
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account_fqdn}",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account_fqdn}",
    get_secret_or_fail(secret_scope, "adls-sp-client-id")
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account_fqdn}",
    get_secret_or_fail(secret_scope, "adls-sp-client-secret")
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_fqdn}",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

jdbc_url = (
    f"jdbc:sqlserver://{sql_server}:1433;"
    f"database={sql_database};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)



In [0]:
dim_customer_df = spark.read.format("delta").load(f"{gold_path}/dim_customer")

(
    dim_customer_df.write
    .format("jdbc")
    .mode("overwrite")
    .option("url", jdbc_url)
    .option("dbtable", "dw.DimCustomer")
    .option("user", sql_user)
    .option("password", sql_password)
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .save()
)

In [0]:
tables = [
    ("dim_customer", "dw.DimCustomer"),
    ("dim_product", "dw.DimProduct"),
    ("dim_store", "dw.DimStore"),
    ("dim_date", "dw.DimDate"),
    ("dim_payment_method", "dw.DimPaymentMethod"),
    ("fact_sales", "dw.FactSales"),
    ("fact_payments", "dw.FactPayments"),
    ("fact_inventory", "dw.FactInventory")
]

available_folders = {item.name.rstrip("/") for item in dbutils.fs.ls(gold_path)}

for delta_folder, sql_table in tables:
    if delta_folder not in available_folders:
        print(f"Skipping {delta_folder}: path not found in {gold_path}")
        continue

    print(f"Loading {delta_folder} into {sql_table}")

    df = spark.read.format("delta").load(f"{gold_path}/{delta_folder}")

    (
        df.write
        .format("jdbc")
        .mode("overwrite")
        .option("url", jdbc_url)
        .option("dbtable", sql_table)
        .option("user", sql_user)
        .option("password", sql_password)
        .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
        .save()
    )

    print(f"Completed {sql_table}")